# MIS 584 — TwinBytes: Scalable Retail Demand Forecasting
## Notebook 1: Data Engineering Pipeline

**Team:** TwinBytes — Ashleigh McNamara, Purva Pandit  
**Course:** MIS 584: Big Data Technologies  
**Dataset:** M5 Forecasting — Accuracy (Walmart, via Kaggle)

---

## What this notebook does

This notebook is the first step in our project pipeline. It takes the raw M5 Walmart sales data and transforms it into a clean, feature-rich dataset ready for exploratory analysis and machine learning.

By the end of this notebook, you will have a Spark table called `m5_features` saved to your session that all other notebooks load from.

---

## Background

The M5 dataset contains daily unit sales for **3,049 products** across **10 Walmart stores** in California, Texas, and Wisconsin, covering approximately 5 years (2011–2016). It comes as three separate files:

| File | Description |
|---|---|
| `sales_train_validation.csv` | Daily sales per product per store — one row per product, one column per day (~1,913 day columns) |
| `calendar.csv` | Date metadata — day of week, month, holidays, SNAP event flags |
| `sell_prices.csv` | Weekly price per product per store |

The sales file is in **wide format** — almost 2,000 columns, one per day. This must be reshaped into **long format** (one row per product-store-day) before it can be used for modeling. This transformation grows the dataset from ~30,000 rows to approximately **58 million rows**, which is why we use Apache Spark.

---

## How to run this notebook

### Step 1 — Upload your data files
Upload all three CSV files to your Colab session:
```
sales_train_validation.csv
calendar.csv
sell_prices.csv
```
Go to the folder icon on the left sidebar → click the upload button → select all three files.

> **Note:** Colab file storage is temporary. If your session disconnects, you will need to re-upload the files and re-run this notebook.

### Step 2 — Run cells in order
Every cell must be run top to bottom. Do not skip cells — each one depends on the previous.

### Step 3 — Watch for the confirmation prints
Each major step prints a row count or confirmation message. Use these to verify things are working before moving on. If a count looks wrong, do not continue — recheck the step above it.

### Step 4 — Save the table at the end
The final cell saves your cleaned dataset as a Spark table called `m5_features`. This is what Notebook 2 (EDA) and Notebook 3 (Modeling) load from. Do not skip this step.

---

## Pipeline overview
```
Raw CSV files
      │
      ▼
Step 1 — Start Spark session
Step 2 — Load CSVs into Spark DataFrames
Step 3 — Melt sales from wide → long format     (~30K rows → ~58M rows)
Step 4 — Join calendar and pricing data
Step 5 — Engineer lag, rolling, and price features
Step 6 — Handle missing values
Step 7 — Save as Spark table: m5_features
```

---

## Output

When this notebook finishes successfully, you will have:

- A Spark table `m5_features` with approximately 58 million rows
- One row per product-store-day combination
- The following columns ready for modeling:

| Column | Description |
|---|---|
| `item_id`, `store_id`, `dept_id`, `cat_id`, `state_id` | Product and store identifiers |
| `date` | Calendar date |
| `sales` | Daily unit sales (target variable) |
| `lag_7`, `lag_14`, `lag_28` | Sales 7, 14, 28 days prior |
| `rolling_avg_7`, `rolling_avg_30`, `rolling_avg_90` | Rolling mean sales over past 7, 30, 90 days |
| `sell_price` | Price of item that week |
| `price_delta` | Change in price vs. prior week |
| `is_event` | 1 if a calendar event occurs that day, else 0 |
| `snap_flag` | 1 if SNAP benefits active in that store's state, else 0 |
| `wday`, `month`, `year` | Temporal features |

---


In [22]:
#Installing Spark
!pip install pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("M5_Retail") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [23]:
#Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
#Loading the M5 Kaggle Data into Spark
sales = spark.read.csv("/content/drive/MyDrive/MIS584_Project/sales_train_validation.csv",
                        header=True, inferSchema=True)
calendar = spark.read.csv("/content/drive/MyDrive/MIS584_Project/calendar.csv",
                           header=True, inferSchema=True)
prices = spark.read.csv("/content/drive/MyDrive/MIS584_Project/sell_prices.csv",
                         header=True, inferSchema=True)

sales.cache()
calendar.cache()
prices.cache()

print(f"Sales rows: {sales.count():,}")
print(f"Sales cols: {len(sales.columns)}")

Sales rows: 30,490
Sales cols: 1919


In [25]:
#Sales data has about 2,000 column names, one per day
#Unpivot the data so that each row is a single observation of one item, one store, one dat, one sales value

#Identify the day columns
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = [c for c in sales.columns if c.startswith("d_")]
print(f"Day columns found: {len(day_cols)}")

#Unpivot the table using Stack()
from pyspark.sql.functions import expr, col

stack_expr = ", ".join([f"'{d}', `{d}`" for d in day_cols])
sales_long = sales.select(
    *id_cols,
    expr(f"stack({len(day_cols)}, {stack_expr}) as (d, sales)")
)

sales_long = sales_long.withColumn("sales", col("sales").cast("float"))
sales_long.cache()

print(f"Long format rows: {sales_long.count():,}")

Day columns found: 1913
Long format rows: 58,327,370


In [26]:
sales_long.show(5)

+--------------------+-------------+---------+-------+--------+--------+---+-----+
|                  id|      item_id|  dept_id| cat_id|store_id|state_id|  d|sales|
+--------------------+-------------+---------+-------+--------+--------+---+-----+
|HOBBIES_1_001_CA_...|HOBBIES_1_001|HOBBIES_1|HOBBIES|    CA_1|      CA|d_1|  0.0|
|HOBBIES_1_001_CA_...|HOBBIES_1_001|HOBBIES_1|HOBBIES|    CA_1|      CA|d_2|  0.0|
|HOBBIES_1_001_CA_...|HOBBIES_1_001|HOBBIES_1|HOBBIES|    CA_1|      CA|d_3|  0.0|
|HOBBIES_1_001_CA_...|HOBBIES_1_001|HOBBIES_1|HOBBIES|    CA_1|      CA|d_4|  0.0|
|HOBBIES_1_001_CA_...|HOBBIES_1_001|HOBBIES_1|HOBBIES|    CA_1|      CA|d_5|  0.0|
+--------------------+-------------+---------+-------+--------+--------+---+-----+
only showing top 5 rows


In [27]:
#Join the long sales table with calendar data and sell_prices using left joins
#Joing calendar on "D" column
calendar_slim = calendar.select(
    "d", "date", "wm_yr_wk", "weekday", "wday",
    "month", "year", "event_name_1", "event_type_1",
    "event_name_2", "event_type_2",
    "snap_CA", "snap_TX", "snap_WI"
)

df = sales_long.join(calendar_slim, on="d", how="left")

In [28]:
#Join prices on the item_id and store_id and wm_yr_wk
#Some early rows will have null sell_price as they were not tracked -- Handled later
df = df.join(prices,
             on=["item_id", "store_id", "wm_yr_wk"],
             how="left")

print(f"Joined rows: {df.count():,}")  # should still be ~58M
df.cache()
df.unpersist()

Joined rows: 58,327,370


DataFrame[item_id: string, store_id: string, wm_yr_wk: int, d: string, id: string, dept_id: string, cat_id: string, state_id: string, sales: float, date: date, weekday: string, wday: int, month: int, year: int, event_name_1: string, event_type_1: string, event_name_2: string, event_type_2: string, snap_CA: int, snap_TX: int, snap_WI: int, sell_price: double]

In [29]:
#Engineer Features
#All features are created using Spark Window functions partitioned by (item_id, store_id) and ordered by date
#This ensures our lag or rolling features never leak outside different time series

#Define the window
from pyspark.sql.functions import (lag, avg, to_date,
    when, col, lit)
from pyspark.sql.window import Window

df = df.withColumn("date", to_date(col("date")))

w = (Window
     .partitionBy("item_id", "store_id")
     .orderBy("date"))

#Lag features (sales N days ago)
for lag_n in [7, 14, 28]:
    df = df.withColumn(f"lag_{lag_n}", lag("sales", lag_n).over(w))

#Rolling average features
for window_size in [7, 30, 90]:
    rolling_w = (Window
                 .partitionBy("item_id", "store_id")
                 .orderBy("date")
                 .rowsBetween(-window_size, -1))
    df = df.withColumn(f"rolling_avg_{window_size}",
                       avg("sales").over(rolling_w))

#Price Delta (week over week)
price_w = (Window
           .partitionBy("item_id", "store_id")
           .orderBy("wm_yr_wk"))
df = df.withColumn("price_lag1", lag("sell_price", 1).over(price_w))
df = df.withColumn("price_delta",
                   col("sell_price") - col("price_lag1"))

#Calendar andd SNAP features
df = df.withColumn("is_event",
    when(col("event_name_1").isNotNull(), 1).otherwise(0))

df = df.withColumn("snap_flag",
    when(col("state_id") == "CA", col("snap_CA"))
    .when(col("state_id") == "TX", col("snap_TX"))
    .when(col("state_id") == "WI", col("snap_WI"))
    .otherwise(0))

In [30]:
#Handling Missing Values
from pyspark.sql.functions import last

fill_w = (Window
          .partitionBy("item_id", "store_id")
          .orderBy("date")
          .rowsBetween(Window.unboundedPreceding, 0))

df = df.withColumn("sell_price",
    last("sell_price", ignorenulls=True).over(fill_w))

#Drop rows where lag features are still null
df_clean = df.dropna(subset=["lag_7", "lag_28", "rolling_avg_90"])
print(f"Clean rows: {df_clean.count():,}")

#Verfiy that all null values are gone
from pyspark.sql.functions import count, when, isnan

null_check = df_clean.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ["lag_7", "lag_28", "rolling_avg_7",
              "sell_price", "snap_flag"]
])
null_check.show()

Clean rows: 57,473,650
+-----+------+-------------+----------+---------+
|lag_7|lag_28|rolling_avg_7|sell_price|snap_flag|
+-----+------+-------------+----------+---------+
|    0|     0|            0|  11783268|        0|
+-----+------+-------------+----------+---------+



In [32]:
#Saving the CSV
from google.colab import drive
drive.mount('/content/drive')\

import os
os.makedirs('/content/drive/MyDrive/MIS584_Project', exist_ok=True)

df_clean.write.mode("overwrite").option("header", "true").csv(
    '/content/drive/MyDrive/MIS584_Project/m5_features'
)
print("Saved successfully")
print("Sample saved to Drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved successfully
Sample saved to Drive
